# Prepare handscan validation datasets

Build **healthy** / **unhealthy** validation sets from `HandscanUnhealthy` archives, crop wire range `[3600, 4000]`, save inspection crops + inference NPZs, and launch a scanner.

**Labeling (from handscan notes)**
- **healthy** ← note exactly `clean plane 1`
- **unhealthy** ← note exactly `streaks on plane 1`

**Kernel:** `env` (`/exp/sbnd/app/users/munjung/env`).

**Outputs** under `handscan_validation/`:
- `manifests/{healthy,unhealthy}.jsonl` — sample metadata
- `cropped/{healthy,unhealthy}/*.npz` — raw wire crop `[3600:4000]` for inspection
- `npz_inference/{healthy,unhealthy}/*.npz` — scaled + padded planes for DDIM inference

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import h5py
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

# ── Configure ────────────────────────────────────────────────────────────────
ARCHIVE_DIR = Path("handscan_archives/offbeamlight_v10_06_00")
UNHEALTHY_PATH = ARCHIVE_DIR / "unhealthy.jsonl"

OUT_ROOT = Path("handscan_validation")
MANIFEST_DIR = OUT_ROOT / "manifests"
CROPPED_DIR = OUT_ROOT / "cropped"
NPZ_DIR = OUT_ROOT / "npz_inference"

WIRE_LO, WIRE_HI = 3600, 4000  # exclusive end → 400 wires
PATCH = 512
# 2nd-induction (plane 1) scale from training-SBND/iterE README / reco_jobs.py
PLANE1_SCALE = 100.0

HEALTHY_NOTE = "clean plane 1"
UNHEALTHY_NOTE = "streaks on plane 1"

IMSHOW_KW = dict(aspect="auto", cmap="bwr", origin="lower", vmin=-15, vmax=15)
FIG_SIZE = (14, 4)

for d in [
    MANIFEST_DIR,
    CROPPED_DIR / "healthy",
    CROPPED_DIR / "unhealthy",
    NPZ_DIR / "healthy",
    NPZ_DIR / "unhealthy",
]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 100
print("Archive:", UNHEALTHY_PATH.resolve())
print("Output :", OUT_ROOT.resolve())

## Filter archive → healthy / unhealthy manifests

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open() as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def note_matches(note: str, target: str) -> bool:
    return note.strip().lower() == target.strip().lower()


def safe_stem(sample_id: str) -> str:
    """Filesystem-safe stem from sample_id like run/file.h5:event."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", sample_id)


def dedupe_by_sample_id(rows: list[dict]) -> list[dict]:
    seen: set[str] = set()
    out: list[dict] = []
    for r in rows:
        sid = r["sample_id"]
        if sid in seen:
            continue
        seen.add(sid)
        out.append(r)
    return out


all_rows = load_jsonl(UNHEALTHY_PATH)
healthy_rows = dedupe_by_sample_id(
    [r for r in all_rows if note_matches(r.get("note", ""), HEALTHY_NOTE)]
)
unhealthy_rows = dedupe_by_sample_id(
    [r for r in all_rows if note_matches(r.get("note", ""), UNHEALTHY_NOTE)]
)

print(f"Archive rows: {len(all_rows)}")
print(f"Healthy   ('{HEALTHY_NOTE}'): {len(healthy_rows)}")
print(f"Unhealthy ('{UNHEALTHY_NOTE}'): {len(unhealthy_rows)}")

pd.DataFrame(
    [
        {"label": "healthy", "n": len(healthy_rows)},
        {"label": "unhealthy", "n": len(unhealthy_rows)},
    ]
)

## Crop, save inspection NPZs, and build inference NPZs

- **Inspection crop:** wires `[WIRE_LO:WIRE_HI]` (raw ADC), shape `(400, n_ticks)`.
- **Inference plane:** divide by `PLANE1_SCALE`, reflect-pad wires to `PATCH`, crop ticks to a multiple of `PATCH`, stored as `reco` with shape `(1, H, W)`.

In [ ]:
def load_waveform(record: dict) -> np.ndarray:
    with h5py.File(record["h5_path"], "r") as hf:
        return np.asarray(hf[record["dataset_path"]], dtype=np.float32)


def crop_wires(arr: np.ndarray, lo: int = WIRE_LO, hi: int = WIRE_HI) -> np.ndarray:
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D waveform, got {arr.shape}")
    if hi > arr.shape[0]:
        raise ValueError(f"Wire hi={hi} exceeds n_wires={arr.shape[0]}")
    return arr[lo:hi, :].astype(np.float32, copy=False)


def make_inference_plane(crop: np.ndarray, scale: float = PLANE1_SCALE, patch: int = PATCH) -> np.ndarray:
    """Scale + pad/crop to (H, W) with H,W multiples of patch (H=patch after pad)."""
    plane = crop.astype(np.float32) / float(scale)
    h, w = plane.shape
    if h > patch:
        plane = plane[:patch, :]
        h = patch
    elif h < patch:
        pad = patch - h
        # Reflect pad on the high-wire side so we do not invent content below WIRE_LO.
        plane = np.pad(plane, ((0, pad), (0, 0)), mode="reflect")
        h = patch
    wc = (w // patch) * patch
    if wc < patch:
        raise ValueError(f"Tick dim {w} smaller than patch {patch}")
    plane = plane[:, :wc]
    return plane.astype(np.float32, copy=False)


def process_split(rows: list[dict], label: str) -> list[dict]:
    manifest: list[dict] = []
    crop_dir = CROPPED_DIR / label
    npz_dir = NPZ_DIR / label
    for rec in tqdm(rows, desc=f"process {label}"):
        stem = safe_stem(rec["sample_id"])
        crop_path = crop_dir / f"{stem}.npz"
        npz_path = npz_dir / f"{stem}.npz"

        full = load_waveform(rec)
        crop = crop_wires(full)
        plane = make_inference_plane(crop)

        np.savez_compressed(
            crop_path,
            crop=crop,
            wire_lo=np.int32(WIRE_LO),
            wire_hi=np.int32(WIRE_HI),
            sample_id=np.asarray(rec["sample_id"]),
            note=np.asarray(rec.get("note", "")),
            label=np.asarray(label),
            h5_path=np.asarray(rec["h5_path"]),
            dataset_path=np.asarray(rec["dataset_path"]),
        )
        np.savez_compressed(
            npz_path,
            reco=plane[np.newaxis, ...].astype(np.float32),  # (1, H, W)
            wire_lo=np.int32(WIRE_LO),
            wire_hi=np.int32(WIRE_HI),
            scale=np.float32(PLANE1_SCALE),
            sample_id=np.asarray(rec["sample_id"]),
            note=np.asarray(rec.get("note", "")),
            label=np.asarray(label),
        )

        entry = {
            **rec,
            "label": label,
            "stem": stem,
            "crop_npz": str(crop_path.resolve()),
            "inference_npz": str(npz_path.resolve()),
            "crop_shape": list(crop.shape),
            "inference_plane_shape": list(plane.shape),
            "wire_lo": WIRE_LO,
            "wire_hi": WIRE_HI,
            "plane1_scale": PLANE1_SCALE,
        }
        manifest.append(entry)
    return manifest


healthy_manifest = process_split(healthy_rows, "healthy")
unhealthy_manifest = process_split(unhealthy_rows, "unhealthy")


def write_manifest(path: Path, rows: list[dict]) -> None:
    with path.open("w") as fh:
        for r in rows:
            fh.write(json.dumps(r) + "\n")


write_manifest(MANIFEST_DIR / "healthy.jsonl", healthy_manifest)
write_manifest(MANIFEST_DIR / "unhealthy.jsonl", unhealthy_manifest)

summary = {
    "healthy_n": len(healthy_manifest),
    "unhealthy_n": len(unhealthy_manifest),
    "wire_lo": WIRE_LO,
    "wire_hi": WIRE_HI,
    "plane1_scale": PLANE1_SCALE,
    "patch": PATCH,
    "example_crop_shape": healthy_manifest[0]["crop_shape"] if healthy_manifest else None,
    "example_inference_shape": healthy_manifest[0]["inference_plane_shape"] if healthy_manifest else None,
}
with (MANIFEST_DIR / "summary.json").open("w") as fh:
    json.dump(summary, fh, indent=2)

print(json.dumps(summary, indent=2))
print("Wrote manifests →", MANIFEST_DIR.resolve())

## Cropped-image scanner

Browse saved crops (`cropped/{healthy,unhealthy}/*.npz`). Use the label filter and next/prev controls.

In [ ]:
class CropScanner:
    """Lightweight browser over saved wire-cropped NPZs."""

    def __init__(self, manifests: dict[str, list[dict]], start_label: str = "unhealthy"):
        self.manifests = manifests
        self.label = start_label if start_label in manifests else next(iter(manifests))
        self.idx = 0
        self._im = None

        self.fig, self.ax = plt.subplots(figsize=FIG_SIZE)
        self.fig.canvas.mpl_connect("key_press_event", self._on_key)
        self.plot_out = widgets.Output()

        self.label_dd = widgets.Dropdown(
            options=list(manifests.keys()),
            value=self.label,
            description="Label:",
            layout=widgets.Layout(width="260px"),
        )
        self.jump_box = widgets.IntText(value=0, description="Go to:", layout=widgets.Layout(width="200px"))
        self.status = widgets.HTML()

        self.btn_prev = widgets.Button(description="Prev (b)", icon="arrow-left")
        self.btn_next = widgets.Button(description="Next (n)", button_style="primary", icon="arrow-right")
        self.btn_jump = widgets.Button(description="Jump")

        self.btn_prev.on_click(lambda _: self.step(-1))
        self.btn_next.on_click(lambda _: self.step(+1))
        self.btn_jump.on_click(lambda _: self.jump(int(self.jump_box.value)))
        self.label_dd.observe(self._on_label, names="value")

        controls = widgets.HBox([self.label_dd, self.btn_prev, self.btn_next, self.jump_box, self.btn_jump])
        self.ui = widgets.VBox([controls, self.status, self.plot_out])
        self.refresh()

    @property
    def rows(self) -> list[dict]:
        return self.manifests[self.label]

    def _on_label(self, change):
        self.label = change["new"]
        self.idx = 0
        self.jump_box.value = 0
        self.refresh()

    def _on_key(self, event):
        if event.key in ("n", "right"):
            self.step(+1)
        elif event.key in ("b", "left"):
            self.step(-1)

    def step(self, delta: int):
        if not self.rows:
            return
        self.idx = int(np.clip(self.idx + delta, 0, len(self.rows) - 1))
        self.jump_box.value = self.idx
        self.refresh()

    def jump(self, idx: int):
        if not self.rows:
            return
        self.idx = int(np.clip(idx, 0, len(self.rows) - 1))
        self.jump_box.value = self.idx
        self.refresh()

    def refresh(self):
        with self.plot_out:
            self.plot_out.clear_output(wait=True)
            if not self.rows:
                self.status.value = f"<b>{self.label}</b>: empty"
                display(self.fig)
                return

            rec = self.rows[self.idx]
            with np.load(rec["crop_npz"], allow_pickle=True) as z:
                data = z["crop"]

            self.ax.clear()
            self._im = self.ax.imshow(data, **IMSHOW_KW)
            self.ax.axhline(3968 - WIRE_LO, color="k", ls=":", lw=1, alpha=0.7)  # plane1/2 boundary in crop
            self.ax.set_title(
                f"[{self.label}] {self.idx + 1}/{len(self.rows)}  {rec['sample_id']}  "
                f"note={rec.get('note', '')!r}  crop={tuple(data.shape)} wires[{WIRE_LO}:{WIRE_HI}]"
            )
            self.ax.set_xlabel("tick")
            self.ax.set_ylabel(f"wire (offset from {WIRE_LO})")
            self.fig.tight_layout()
            display(self.fig)

            self.status.value = (
                f"<b>{self.label}</b> idx={self.idx} &nbsp;|&nbsp; "
                f"sample_id=<code>{rec['sample_id']}</code> &nbsp;|&nbsp; "
                f"crop NPZ=<code>{Path(rec['crop_npz']).name}</code>"
            )

    def close(self):
        plt.close(self.fig)


if "crop_scanner" in globals() and hasattr(crop_scanner, "close"):
    crop_scanner.close()

crop_scanner = CropScanner(
    {"healthy": healthy_manifest, "unhealthy": unhealthy_manifest},
    start_label="unhealthy",
)
display(crop_scanner.ui)

## Quick table of prepared samples

In [ ]:
df = pd.DataFrame(healthy_manifest + unhealthy_manifest)[
    ["label", "sample_id", "note", "crop_shape", "inference_plane_shape", "stem"]
]
display(df)
print("healthy crops   :", len(list((CROPPED_DIR / "healthy").glob("*.npz"))))
print("unhealthy crops :", len(list((CROPPED_DIR / "unhealthy").glob("*.npz"))))
print("healthy npz     :", len(list((NPZ_DIR / "healthy").glob("*.npz"))))
print("unhealthy npz   :", len(list((NPZ_DIR / "unhealthy").glob("*.npz"))))